# 第 37 课：微调诊断与泛化评测
## 过拟合、数据泄漏、旧能力退步与配对不确定性

上一课把量化基座和 LoRA 训练接通了。这一课问：**这次更新值得保留吗？证据是什么？**

本讲义可以独立运行，不依赖前几课的代码。默认只用 Python 标准库和 PyTorch，CPU 即可完成，不下载模型或数据。

**本课不提供虚构的 Qwen 成绩。** 数字来自实际训练的小型神经网络；人为评分记录另行标记。真实实验的原始回答，可以用本课脚本从第 35、36 课目录读取，重新评分而不重新加载模型。

学习方式：先预测，再运行；看到结果后，写出“这个结果说明什么”和“它不能说明什么”。

## 0. 学习目标与操作约定

| 掌握内容 | 可检验的产出 |
|---|---|
| 识别过拟合 | 区分训练、开发和测试；按开发集选轮；保留最终轮作预先规定的对照 |
| 排查泄漏 | 检查来源跨侧、相同提示冲突；近重复只提出人工复核候选 |
| 检查旧能力 | 独立记录目标任务、保留任务和分布变化任务，不能用一个平均数隐藏退步 |
| 比较两个模型 | 按 ID 配对，保留缺失分母，列出改善与退步题 |
| 理解不确定性 | 同一来源的变体成组重采样；不把退化区间当作效果相同证明 |
| 分析上一课结果 | 读取原始答案，重新计算分数；缺少旧能力数据就明确说缺少 |

建议分两轮学习：第一轮到神经网络实验，第二轮做统计、文件分析与练习。这里不使用训练耗时作为 GPU 性能结论。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第37课：微调诊断与泛化评测。
默认只用 CPU/PyTorch：来源泄漏、含噪小样本过拟合、冻结基座的LoRA行为退化。
所有神经网络结果均由本程序实际训练；手工评分样例明确另行标注。
--analyze-run 读取第35/36课原始结果，不加载模型，不执行被读取目录中的代码。
报告型检查不能证明任意任务无泄漏/无退化；默认实验不是LLM能力基准。
两份课程材料互不依赖。来源和详细解释见同名Notebook。
"""
from __future__ import annotations

import argparse
import copy
import hashlib
import json
import math
import platform
import random
import re
import sys
import tempfile
import unicodedata
from collections import defaultdict
from datetime import datetime, timezone
from difflib import SequenceMatcher
from pathlib import Path
from typing import Callable

import torch
from torch import nn
from torch.nn import functional as F

FORMAT = "lesson37-diagnostics-v1"
SEED = 37
SOURCES = {
    "leakage": "https://scikit-learn.org/stable/common_pitfalls.html",
    "groups": "https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html",
    "forgetting": "https://arxiv.org/abs/2401.05605",
    "evaluation_statistics": "https://www.anthropic.com/research/statistical-approach-to-model-evals",
    "reproducibility": "https://docs.pytorch.org/docs/stable/notes/randomness.html",
    "json": "https://docs.python.org/3/library/json.html",
}


def require(condition, message: str) -> None:
    if not bool(condition):
        raise ValueError(message)


def fingerprint(value) -> str:
    raw = json.dumps(value, ensure_ascii=False, sort_keys=True,
                     separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def unique_object(pairs):
    result = {}
    for key, value in pairs:
        require(key not in result, f"JSON存在重复键：{key}")
        result[key] = value
    return result


def reject_constant(value):
    raise ValueError(f"JSON不接受非有限数：{value}")


def strict_json(text: str):
    require(isinstance(text, str), "JSON输入必须是字符串")
    return json.loads(text, object_pairs_hook=unique_object, parse_constant=reject_constant)


def read_json(path: Path):
    return strict_json(Path(path).read_text(encoding="utf-8"))


def read_jsonl(path: Path) -> list[dict]:
    out = []
    for line_no, line in enumerate(Path(path).read_text(encoding="utf-8").splitlines(), 1):
        require(bool(line.strip()), f"{path}:{line_no}为空行")
        record = strict_json(line)
        require(isinstance(record, dict), f"{path}:{line_no}不是JSON对象")
        out.append(record)
    return out


def write_json(path: Path, value) -> None:
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                    allow_nan=False), encoding="utf-8")


def new_run(root, prefix="offline") -> Path:
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    path = root / f"{prefix}_{stamp}"
    path.mkdir(exist_ok=False)
    return path


def state_copy(model: nn.Module) -> dict:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def tensor_hash(tensor: torch.Tensor) -> str:
    t = tensor.detach().cpu().contiguous()
    # 本课仅存小模型；不依赖NumPy字节接口。
    return fingerprint({"dtype": str(t.dtype), "shape": list(t.shape), "data": t.tolist()})


print("Python：", platform.python_version())
print("PyTorch：", torch.__version__)
print("本课计算设备：CPU；不调用 CUDA")
torch.set_num_threads(1)

Python： 3.13.5
PyTorch： 2.10.0+cpu
本课计算设备：CPU；不调用 CUDA


上一个单元是可展开的公共工具定义。它只导入现有库，不安装任何东西。

## 1. 先写一张实验卡

在看结果之前确定：训练用什么、用什么选择模型、最终评什么、失败怎样计分、允许多少退步。

**开发集可以用来选轮，测试集不可以。** 多次观察测试错题并调参之后，它就变成开发/回归材料，应另外准备最终留出数据。预处理规则也只能在训练侧学习。[官方数据泄漏说明](https://scikit-learn.org/stable/common_pitfalls.html)

下面只是本课的教学协议，不是任何应用的统一准入标准。

In [2]:
protocol = {
    "selection_split": "dev",
    "final_suites": ["target_test", "retention_test", "shift_test"],
    "selection_rule": "min target_dev_loss subject to retention_dev_accuracy drop <= 0.02",
    "missing_answer": "failure, still in denominator",
    "primary_score": "strict_success",
    "uncertainty_unit": "source_group",
}
print(json.dumps(protocol, ensure_ascii=False, indent=2))

{
  "selection_split": "dev",
  "final_suites": [
    "target_test",
    "retention_test",
    "shift_test"
  ],
  "selection_rule": "min target_dev_loss subject to retention_dev_accuracy drop <= 0.02",
  "missing_answer": "failure, still in denominator",
  "primary_score": "strict_success",
  "uncertainty_unit": "source_group"
}


## 2. 来源切分：评到的是“新问法”还是“新来源”？

如果同一订单的两种问法分别进入训练与测试，模型可能记住订单，而不是学会处理新的订单。不同任务可以有不同泛化目标；**不能把已见来源上的新问法成绩，当作新来源泛化成绩。**

按来源划分的单位可以是原始文档、患者、代码仓库、会话或订单。`group_id` 必须反映真实来源，不是为了通过检查而随便填不同值。[GroupShuffleSplit 官方说明](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html)

本节用一个真实执行的查表程序，不用神经网络，隔离“按来源记忆”这个机制。

In [3]:
def record_prompt(record: dict) -> list[dict]:
    require(isinstance(record, dict), "记录必须是字典")
    for key in ("id", "group_id"):
        require(isinstance(record.get(key), str) and bool(record[key].strip()), f"{key}必须非空")
    messages = record.get("messages")
    require(isinstance(messages, list) and len(messages) >= 2, "至少一轮完整对话")
    for m in messages:
        require(isinstance(m, dict) and set(m) == {"role", "content"}, "消息字段必须为role/content")
        require(m["role"] in {"system", "user", "assistant"}, "本课只处理纯文本聊天")
        require(isinstance(m["content"], str) and bool(m["content"].strip()), "消息不能为空")
        m["content"].encode("utf-8", errors="strict")
    start = int(messages[0]["role"] == "system")
    require((len(messages) - start) % 2 == 0, "消息必须是完整问答")
    for i, m in enumerate(messages[start:]):
        require(m["role"] == ("user" if i % 2 == 0 else "assistant"), "问答角色顺序错误")
    # 去掉最后的参考答案，不把答案当成去重提示或模型输入。
    return copy.deepcopy(messages[:-1])


def group_disjoint(parts: dict[str, list[dict]]) -> bool:
    require(isinstance(parts, dict) and len(parts) >= 2, "至少两个划分")
    ownership, ids = {}, set()
    for split, records in parts.items():
        require(isinstance(records, list) and bool(records), "划分不能为空")
        for row in records:
            require(isinstance(row, dict), "记录不是字典")
            rid, gid = row.get("id"), row.get("group_id")
            require(isinstance(rid, str) and bool(rid) and isinstance(gid, str) and bool(gid), "ID/来源非法")
            require(rid not in ids, f"重复记录ID：{rid}")
            ids.add(rid)
            require(gid not in ownership or ownership[gid] == split,
                    f"同一来源跨划分：{gid}")
            ownership[gid] = split
    return True


def normalized_prompt_for_audit(prompt) -> str:
    # 仅生成审计签名，不改变真正交给模型的内容，不自动删除数据。
    # 压缩空白可能改变代码缩进语义，因此只作为警告。
    return "\n".join(m["role"] + ":" + re.sub(r"\s+", " ",
                        unicodedata.normalize("NFC", m["content"])).strip() for m in prompt)


def audit_splits(parts: dict[str, list[dict]], threshold=0.96, max_warnings=30) -> dict:
    require(0 < threshold <= 1, "近重复阈值不合法")
    require(type(max_warnings) is int and max_warnings >= 0, "警告上限无效")
    group_disjoint(parts)
    exact, normalized, all_rows = {}, {}, []
    hard, warnings = [], []
    total_candidates = 0
    for split, rows in parts.items():
        for row in rows:
            prompt = record_prompt(row)
            key = fingerprint(prompt)
            target = row["messages"][-1]["content"]
            entry = {"split": split, "id": row["id"], "group_id": row["group_id"],
                     "prompt": prompt, "target": target, "normalized": normalized_prompt_for_audit(prompt)}
            for old in exact.get(key, []):
                if old["split"] != split:
                    hard.append({"kind": "exact_prompt_overlap", "left": old["id"], "right": row["id"],
                                 "conflicting_answer": old["target"] != target})
            exact.setdefault(key, []).append(entry)
            all_rows.append(entry)
    # 明确O(n^2)：课程小数据，不适合百万条数据；真实大数据应预筛候选。
    for i, left in enumerate(all_rows):
        for right in all_rows[i+1:]:
            if left["split"] == right["split"] or left["prompt"] == right["prompt"]:
                continue
            score = SequenceMatcher(None, left["normalized"], right["normalized"], autojunk=False).ratio()
            if score >= threshold:
                total_candidates += 1
                if len(warnings) < max_warnings:
                    warnings.append({"kind": "near_duplicate_candidate", "left": left["id"],
                                     "right": right["id"], "similarity": score,
                                     "action": "人工检查；不自动删除，数字差异可能改变正确答案"})
    return {"status": "failed" if hard else "passed_with_scope_limits", "hard_errors": hard,
            "near_duplicate_candidates": warnings, "candidate_count": total_candidates,
            "candidates_truncated": total_candidates > len(warnings),
            "sizes": {k: len(v) for k, v in parts.items()},
            "groups": {k: len({r['group_id'] for r in v}) for k, v in parts.items()},
            "limits": "仅显式来源与末轮精确/表面近重复；不能证明语义无泄漏或基座预训练未见过数据"}


def leakage_demo() -> dict:
    # 随机/平衡标签只由文档身份决定；不存在可迁移的内容规则。
    records = []
    for g in range(12):
        for variant in range(2):
            prompt = (f"Document P{g:02d}: report the stored class." if variant == 0
                      else f"For document P{g:02d}, return its registered category.")
            records.append({"id": f"p{g:02d}-v{variant}", "group_id": f"p{g:02d}",
                "messages": [{"role": "user", "content": prompt},
                             {"role": "assistant", "content": str(g % 2)}]})
    bad = {"train": records[::2], "test": records[1::2]}
    good = {"train": [r for r in records if int(r['group_id'][1:]) < 8],
            "test": [r for r in records if int(r['group_id'][1:]) >= 8]}
    def measure(parts):
        memory = {}
        for r in parts["train"]:
            key = re.search(r"P\d+", r["messages"][0]["content"])[0]
            memory[key] = r["messages"][-1]["content"]
        rows = []
        for r in parts["test"]:
            key = re.search(r"P\d+", r["messages"][0]["content"])[0]
            prediction = memory.get(key, "0")  # 未见来源采用预先规定的常数基线。
            rows.append({"id": r["id"], "seen_source": key in memory,
                         "prediction": prediction, "expected": r["messages"][-1]["content"]})
        return {"count": len(rows), "accuracy": sum(r['prediction']==r['expected'] for r in rows)/len(rows),
                "seen_source_rate": sum(r['seen_source'] for r in rows)/len(rows), "rows": rows}
    failed = False
    try:
        group_disjoint(bad)
    except ValueError:
        failed = True
    return {"kind": "actual_dictionary_lookup_not_neural_model", "bad_split_detected": failed,
            "overlapping_sources": measure(bad), "unseen_sources": measure(good),
            "limits": "两个划分测量不同泛化对象；不把分数差当作某个真实模型的泄漏影响估计"}

In [4]:
leakage = leakage_demo()
for name in ("overlapping_sources", "unseen_sources"):
    item = leakage[name]
    print(name, "准确率：", item["accuracy"], "已见来源比例：", item["seen_source_rate"])
print("来源审计发现错误切分：", leakage["bad_split_detected"])
print("限制：", leakage["limits"])

overlapping_sources 准确率： 1.0 已见来源比例： 1.0
unseen_sources 准确率： 0.5 已见来源比例： 0.0
来源审计发现错误切分： True
限制： 两个划分测量不同泛化对象；不把分数差当作某个真实模型的泄漏影响估计


### 观察与解释

上例的查询文本中含有文档编号；查表器从训练数据记住编号到类别的对应。训练侧没有该编号时，一律猜预先规定的类别 0。

重叠来源的评分可以达到 100%，未见来源则只有常数基线表现。**这不是两个同分布测试集上的公平模型对比，而是两种评估问题不同的演示。**

没有来源重叠，也不等于完全排除了泄漏：可能还有复制改写、时间信息、共享模板、检索内容带答案、甚至基座预训练见过评测文本。当前代码没有能力证明最后一项。

In [5]:
# ID和来源不同，但输入提示完全相同，也要检查。
example_parts = {
    "train": [{"id":"a", "group_id":"source-A", "messages":[
        {"role":"user","content":"Order 12: return the quantity."},
        {"role":"assistant","content":'{"quantity":1}'}]}],
    "test": [{"id":"b", "group_id":"source-B", "messages":[
        {"role":"user","content":"Order 12: return the quantity."},
        {"role":"assistant","content":'{"quantity":2}'}]}],
}
audit = audit_splits(example_parts)
print(audit["hard_errors"])
print("原始文本没有被重写。")

[{'kind': 'exact_prompt_overlap', 'left': 'a', 'right': 'b', 'conflicting_answer': True}]
原始文本没有被重写。


### 精确重复与近重复，不要使用相同的处置方式

本课对跨侧来源重复直接报错，对末轮精确提示重叠生成硬错误。近重复只列为候选，**不自动删除**：订单 12 与订单 13 的文本很相似，正确答案却可能不同。

审计用 NFC 和空白压缩来发现表面相似；它不改变实际输入。压缩空白会改变代码缩进含义，因此尤其不能把审计规范化直接用作训练预处理。

本课近重复检查是 $O(n^2)$ 的小数据版本，不适合百万条数据。

In [6]:
near_parts = copy.deepcopy(example_parts)
near_parts["test"][0]["messages"][0]["content"] = "Order  12: return the quantity."
near = audit_splits(near_parts)
print("硬错误：", near["hard_errors"])
print("人工复核候选：", near["near_duplicate_candidates"])
print("候选报告可能截短，完整候选数：", near["candidate_count"])

硬错误： []
人工复核候选： [{'kind': 'near_duplicate_candidate', 'left': 'a', 'right': 'b', 'similarity': 1.0, 'action': '人工检查；不自动删除，数字差异可能改变正确答案'}]
候选报告可能截短，完整候选数： 1


## 3. loss 曲线：什么现象值得调查？

| 现象 | 值得排查的方向，但不是唯一结论 |
|---|---|
| 训练、开发都高 | 标签/梯度断路、学习率、容量、优化步数、任务难度 |
| 训练下降，开发先降后升 | 过拟合、训练噪声、分布变化；先核对两侧统计口径 |
| 两侧 loss 都低，生成任务仍失败 | teacher forcing 与自由生成差异、格式/EOS、评分规则 |
| 开发很好，最终测试很差 | 反复调开发集、来源泄漏、任务分布差异、小样本波动 |
| 目标任务提高，旧任务下降 | adapter 开启时的行为退化；还要检查模板、解码和精度是否一致 |

在比较之前，先保证两侧的标签屏蔽、有效 token 分母、模板和推理设置一致。单看一条训练曲线不能诊断所有问题。

### 实验：含噪 XOR 的小样本过拟合

二维输入、二分类 MLP；训练 48 条，开发 256 条，测试 256 条。三侧独立采样，并都以 15% 概率翻转标签。不是语言模型，也不是预训练模型。

固定训练 500 步，每 10 步计算开发集，按开发交叉熵选最佳。测试集只在训练/选择结束后使用；最后一步作为预先规定的对照，不通过测试成绩反过来选轮。

In [7]:
def choose_dev_checkpoint(history: list[dict], baseline_retention=1.0, max_drop=0.02) -> int | None:
    require(isinstance(history, list) and bool(history), "历史为空")
    require(math.isfinite(baseline_retention) and 0 <= baseline_retention <= 1, "基准保留准确率无效")
    require(math.isfinite(max_drop) and 0 <= max_drop <= 1, "退步阈值无效")
    steps, eligible = set(), []
    for row in history:
        require(row.get("split") == "dev", "只能用dev选择；不能把train/test/OOD作为选轮输入")
        step, loss, retain = row.get("step"), row.get("new_loss"), row.get("retention_accuracy")
        require(type(step) is int and step >= 0 and step not in steps, "step重复/非法")
        steps.add(step)
        require(type(loss) in (int,float) and math.isfinite(loss) and loss >= 0, "新任务dev loss无效")
        require(type(retain) in (int,float) and math.isfinite(retain) and 0 <= retain <= 1, "保留集准确率无效")
        if retain >= baseline_retention-max_drop:
            eligible.append((loss, step))
    return min(eligible)[1] if eligible else None


def noisy_xor(seed: int, count: int):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(count, 2, generator=g)
    y = (x[:, 0]*x[:, 1] > 0).long()
    flip = torch.rand(count, generator=g) < .15
    return x, y ^ flip.long()


def make_classifier():
    return nn.Sequential(nn.Linear(2,64), nn.Tanh(), nn.Linear(64,64), nn.Tanh(), nn.Linear(64,2))


@torch.no_grad()
def classification_metrics(model, data) -> dict:
    previous = model.training
    model.eval()
    try:
        x, y = data
        logits = model(x)
        return {"loss": float(F.cross_entropy(logits, y)),
                "accuracy": float((logits.argmax(-1)==y).double().mean()),
                "count": len(y), "predictions": logits.argmax(-1).tolist(), "targets": y.tolist()}
    finally:
        model.train(previous)


def fit_overfit_demo(steps=500) -> tuple[dict, dict]:
    require(type(steps) is int and steps >= 10 and steps % 10 == 0, "steps必须是至少10的10倍数")
    train, dev = noisy_xor(3701,48), noisy_xor(3702,256)
    # 这里不构造test；训练/选择结束后才产生独立固定测试样本。
    torch.manual_seed(SEED)
    model = make_classifier()
    optimizer = torch.optim.Adam(model.parameters(), lr=.015)
    history, best_loss, best_step, best_state = [], math.inf, None, None
    for step in range(steps+1):
        if step:
            model.train(); optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(train[0]), train[1])
            loss.backward(); optimizer.step()
        if step % 10 == 0:
            tr, dv = classification_metrics(model, train), classification_metrics(model, dev)
            if dv['loss'] < best_loss:
                best_loss, best_step, best_state = dv['loss'], step, state_copy(model)
            history.append({"step": step, "train_loss": tr['loss'], "train_accuracy": tr['accuracy'],
                            "dev_loss": dv['loss'], "dev_accuracy": dv['accuracy']})
    last_state = state_copy(model)
    test = noisy_xor(3703,256)
    last_metrics = classification_metrics(model, test)
    model.load_state_dict(best_state)
    selected_metrics = classification_metrics(model, test)
    report = {"kind": "trained_small_MLP_not_LLM", "noise_probability": .15,
              "data": {"train":48, "dev":256, "test":256},
              "selection": "每10步仅按dev交叉熵；并列保留更早轮次；最终轮是预先规定的对照，不据test改选",
              "selected_step": best_step, "selected_dev_loss": best_loss,
              "history": history, "selected_test": selected_metrics, "last_test": last_metrics,
              "parameters": sum(p.numel() for p in model.parameters()),
              "limits": "一个固定种子的含噪XOR教学实验；loss差距不是任何训练问题的唯一诊断"}
    return report, {"selected": best_state, "last": last_state}

In [8]:
overfit_report, overfit_states = fit_overfit_demo()
print("参数量：", overfit_report["parameters"])
print("开发集所选步数：", overfit_report["selected_step"])
for row in overfit_report["history"]:
    if row["step"] in {0,10,50,100,250,500}:
        print(row)
print("所选测试loss：", overfit_report["selected_test"]["loss"])
print("最终测试loss：", overfit_report["last_test"]["loss"])

参数量： 4482
开发集所选步数： 10
{'step': 0, 'train_loss': 0.6890919804573059, 'train_accuracy': 0.5625, 'dev_loss': 0.7080051302909851, 'dev_accuracy': 0.46875}
{'step': 10, 'train_loss': 0.4689011871814728, 'train_accuracy': 0.7916666666666666, 'dev_loss': 0.5432079434394836, 'dev_accuracy': 0.765625}
{'step': 50, 'train_loss': 0.32265666127204895, 'train_accuracy': 0.8541666666666666, 'dev_loss': 0.7085774540901184, 'dev_accuracy': 0.70703125}
{'step': 100, 'train_loss': 0.10697191953659058, 'train_accuracy': 1.0, 'dev_loss': 1.098222017288208, 'dev_accuracy': 0.7265625}
{'step': 250, 'train_loss': 0.00444362498819828, 'train_accuracy': 1.0, 'dev_loss': 1.9340459108352661, 'dev_accuracy': 0.6953125}
{'step': 500, 'train_loss': 0.0009719692170619965, 'train_accuracy': 1.0, 'dev_loss': 2.2527918815612793, 'dev_accuracy': 0.67578125}
所选测试loss： 0.515342116355896
最终测试loss： 1.949211835861206


### 怎样解释这次结果？

正常运行会看到训练 loss 接近零，但开发 loss 明显上升。它说明这个固定实验中，继续拟合训练样本并未持续改善留出数据。

**不能把第 10 步作为其他任务的最佳步数。** 本课固定种子让你能核对程序，不是保证跨版本、平台和硬件逐位一致。[PyTorch 复现说明](https://docs.pytorch.org/docs/stable/notes/randomness.html)

比较 loss 和准确率也很有价值：一个模型越来越自信地答错，会显著增加交叉熵，即使准确率变化不大。

In [9]:
for name in ("selected_test", "last_test"):
    r = overfit_report[name]
    print(name, "交叉熵=", round(r["loss"],6), "准确率=", r["accuracy"])
# 这里是读取已经锁定生成的测试结果，不执行重新选择。
print("不以测试成绩修改 selected_step：", overfit_report["selected_step"])

selected_test 交叉熵= 0.515342 准确率= 0.78515625
last_test 交叉熵= 1.949212 准确率= 0.76171875
不以测试成绩修改 selected_step： 10


## 4. LoRA 的基座没改，为什么旧能力还能退步？

冻结参数不等于冻结模型行为。adapter 开启时用的是 $W_0+BA$，不是 $W_0$。已有研究也在 LoRA 微调中观察到旧能力退化；它不是只训练少量参数就能自动排除的问题。[研究论文](https://arxiv.org/abs/2401.05605)

本课用一个刻意构造的最小反例，不把真实语言模型的复杂行为藏在大网络里：

- 基座为 $W_0=[2,0]$，不是预训练而来的大模型。
- 旧任务输入 $[u,0]$，判断 $u$ 的正负。
- 新任务训练输入 $[-v,v]$，判断第二个特征 $v$ 的正负。
- 分布变化测试输入 $[3v,v]$，目标仍是 $v$ 的正负。

初始基座擅长旧任务，但在新任务训练分布上总是选错。新任务训练时的负相关特征制造了捷径。

**这里一个输出的 rank=1 增量能表示任意 1×2 更新；这个实验不演示 LoRA 节省参数，只演示冻结不等于行为不变。**

In [10]:
def index_rows(rows: list[dict], allowed: set[str]) -> dict[str, dict]:
    require(isinstance(rows, list), "响应集合必须是列表")
    result = {}
    for row in rows:
        require(isinstance(row, dict), "响应必须是字典")
        rid = row.get("id")
        require(isinstance(rid, str) and rid in allowed, "响应包含额外/非法ID")
        require(rid not in result, f"响应ID重复：{rid}")
        result[rid] = row
    return result

def validate_plan(plan: list[dict]) -> set[str]:
    require(isinstance(plan, list) and bool(plan), "计划题目为空")
    ids = set()
    for row in plan:
        require(isinstance(row, dict), "计划行必须是字典")
        for key in ("id", "group_id"):
            require(isinstance(row.get(key), str) and bool(row[key]), f"计划{key}非法")
        require(row["id"] not in ids, "计划ID重复")
        ids.add(row["id"])
    return ids

def paired_outcomes(plan: list[dict], before: list[dict], after: list[dict]) -> dict:
    ids = validate_plan(plan)
    a, b = index_rows(before, ids), index_rows(after, ids)
    for row in list(a.values())+list(b.values()):
        require(type(row.get("success")) is bool, "success必须是布尔值")
    groups = {"improved": [], "regressed": [], "both_correct": [], "both_wrong": []}
    rows = []
    for case in plan:
        rid = case["id"]
        x = a.get(rid, {}).get("success", False)
        y = b.get(rid, {}).get("success", False)
        kind = "both_correct" if x and y else "both_wrong" if not x and not y else "improved" if y else "regressed"
        groups[kind].append(rid)
        rows.append({"id": rid, "group_id": case["group_id"], "before": x, "after": y,
                     "delta": int(y)-int(x), "before_missing": rid not in a, "after_missing": rid not in b})
    n = len(plan)
    return {"planned": n, "base_rate": sum(r['before'] for r in rows)/n,
            "adapter_rate": sum(r['after'] for r in rows)/n,
            "delta": sum(r['delta'] for r in rows)/n, **groups, "rows": rows}

class SharedLoRA(nn.Module):
    """一个输出的线性分类器。rank=1可表达任意1x2增量，本例不展示参数节省。"""
    def __init__(self):
        super().__init__()
        self.register_buffer("base_weight", torch.tensor([[2.,0.]], dtype=torch.float64))
        self.a = nn.Parameter(torch.tensor([[1.,-1.]], dtype=torch.float64)/math.sqrt(2))
        self.b = nn.Parameter(torch.zeros(1,1, dtype=torch.float64))

    def forward(self, x, disable_adapter=False):
        base = F.linear(x, self.base_weight)
        return base if disable_adapter else base+F.linear(F.linear(x,self.a),self.b)

    def effective_weight(self):
        return self.base_weight+self.b@self.a


def domain_data(seed: int, count: int, kind: str):
    require(kind in {"target","retention","shift"}, "未知分布")
    g = torch.Generator().manual_seed(seed)
    v = (.5+torch.rand(count,1,generator=g,dtype=torch.float64))*torch.where(
        torch.rand(count,1,generator=g)>.5, 1., -1.)
    if kind == "target":
        x = torch.cat([-v,v], dim=1)
    elif kind == "retention":
        x = torch.cat([v,torch.zeros_like(v)], dim=1)
    else:
        x = torch.cat([3*v,v], dim=1)
    return x, (v>0).double()


@torch.no_grad()
def binary_metrics(model, data, disable_adapter=False) -> dict:
    x,y = data
    z = model(x, disable_adapter=disable_adapter)
    return {"loss": float(F.binary_cross_entropy_with_logits(z,y)),
            "accuracy": float(((z>=0)==y.bool()).double().mean()),
            "count": len(y), "predictions": (z[:,0]>=0).long().tolist(),
            "targets": y[:,0].long().tolist()}


def fit_adapter_branch(train_dev: dict, replay: bool, steps=200) -> tuple[dict,dict]:
    require(set(train_dev)=={"target_train","retention_train","target_dev","retention_dev"},
            "训练函数只接受训练/开发数据，不能传test或shift")
    require(type(replay) is bool and type(steps) is int and steps>0, "训练设置非法")
    model = SharedLoRA(); frozen = tensor_hash(model.base_weight)
    optimizer = torch.optim.Adam([model.a,model.b],lr=.03)
    history, target_best, guarded_best = [], (math.inf,None,None), (math.inf,None,None)
    for step in range(steps+1):
        if step:
            optimizer.zero_grad(set_to_none=True)
            x,y = train_dev['target_train']
            loss = F.binary_cross_entropy_with_logits(model(x),y)
            if replay:
                x_old,y_old = train_dev['retention_train']
                loss = loss+F.binary_cross_entropy_with_logits(model(x_old),y_old)
            loss.backward(); optimizer.step()
        # 为了观察选择行为，本小模型每步验证。没有测试数据参与。
        new = binary_metrics(model,train_dev['target_dev'])
        old = binary_metrics(model,train_dev['retention_dev'])
        row = {"split":"dev", "step":step, "new_loss":new['loss'],
               "new_accuracy":new['accuracy'], "retention_accuracy":old['accuracy'],
               "retention_loss":old['loss']}
        history.append(row)
        if new['loss']<target_best[0]:
            target_best = (new['loss'],step,state_copy(model))
        if old['accuracy']>=.98 and new['loss']<guarded_best[0]:
            guarded_best = (new['loss'],step,state_copy(model))
    require(tensor_hash(model.base_weight)==frozen, "冻结基座发生变化")
    require(choose_dev_checkpoint(history,1.0,.02)==guarded_best[1], "保留集选轮逻辑不一致")
    return {"replay":replay,"history":history,"target_selected_step":target_best[1],
            "guarded_selected_step":guarded_best[1],"base_unchanged":True,
            "updates":steps,"target_examples_per_step":len(train_dev['target_train'][1]),
            "replay_examples_per_step":len(train_dev['retention_train'][1]) if replay else 0}, \
           {"target_selected":target_best[2],"guarded_selected":guarded_best[2]}


def forgetting_demo(steps=200) -> tuple[dict,dict]:
    train_dev={"target_train":domain_data(1,128,"target"),"retention_train":domain_data(2,128,"retention"),
               "target_dev":domain_data(3,64,"target"),"retention_dev":domain_data(4,64,"retention")}
    branches,states={},{}
    for label,replay in (("target_only",False),("with_replay",True)):
        branches[label],states[label]=fit_adapter_branch(train_dev,replay,steps)
    # 所有训练/选择结束后，才构造和使用三个锁定测试集合。
    tests={"target":domain_data(5,256,"target"),"retention":domain_data(6,256,"retention"),
           "shift":domain_data(7,256,"shift")}
    models={"base":SharedLoRA()}
    for name in branches:
        for selection in ("target_selected","guarded_selected"):
            m=SharedLoRA(); m.load_state_dict(states[name][selection]);models[name+"_"+selection]=m
    results={}
    for label,model in models.items():
        results[label]={suite:binary_metrics(model,data) for suite,data in tests.items()}
        results[label]["effective_weight"]=model.effective_weight().detach().tolist()
        require(torch.equal(model.base_weight,models['base'].base_weight), "基座权重改变")
        for data in tests.values():
            require(torch.equal(model(data[0],True),models['base'](data[0],True)), "关闭adapter未恢复基座输出")
    comparisons={}
    for label in results:
        if label=="base":continue
        comparisons[label]={}
        for suite in tests:
            base,after=results['base'][suite],results[label][suite]
            plan=[{"id":f"{suite}-{i}","group_id":f"{suite}-{i}"} for i in range(len(base['targets']))]
            b=[{"id":r['id'],"success":p==y} for r,p,y in zip(plan,base['predictions'],base['targets'])]
            a=[{"id":r['id'],"success":p==y} for r,p,y in zip(plan,after['predictions'],after['targets'])]
            comparisons[label][suite]=paired_outcomes(plan,b,a)
    return {"kind":"trained_shared_linear_LoRA_mechanism_not_LLM","branches":branches,"test":results,
            "paired":comparisons,"disabled_adapter_restores_base":True,
            "guardrail":{"split":"dev","max_retention_accuracy_drop":.02,"primary":"minimize_target_dev_loss"},
            "limits":["解析构造的线性基座，不是预训练LLM；单输出rank=1不是低参数量优势演示",
                      "故意构造目标训练相关性与shift分布；100%/0%不是实际微调幅度预测",
                      "回放使用额外旧任务训练样本，相同步数但更多监督和计算；不声称等算力",
                      "仅评价三个预先固定集合；不能证明通用能力或安全性保留"]}, states

In [11]:
forgetting_report, adapter_states = forgetting_demo()
for name in ("base", "target_only_target_selected", "with_replay_target_selected"):
    row = forgetting_report["test"][name]
    print(name, "有效权重：", row["effective_weight"])
    print({suite: row[suite]["accuracy"] for suite in ("target","retention","shift")})
print("关闭adapter可恢复基座：", forgetting_report["disabled_adapter_restores_base"])

base 有效权重： [[2.0, 0.0]]
{'target': 0.0, 'retention': 1.0, 'shift': 1.0}
target_only_target_selected 有效权重： [[-2.1810015669722134, 4.181001566972213]]
{'target': 1.0, 'retention': 0.0, 'shift': 0.0}
with_replay_target_selected 有效权重： [[3.952366113789206, 8.552592183007832]]
{'target': 1.0, 'retention': 1.0, 'shift': 1.0}
关闭adapter可恢复基座： True


### 注意“知识被永久抹掉”和“开启 adapter 时行为退化”的区别

本课始终核对基座逐位不变。关闭 adapter，基座输出恢复。这里展示的是**adapter 开启时的功能性退化**，不是声称冻结权重里的知识被永久清除。

回放分支加入了额外的旧任务训练样本，目标为：

$$L=L_{new}+L_{replay}$$

它从不使用旧任务测试集来训练。两个分支优化步数相同，但回放分支处理更多训练样本、做更多计算；不能宣称这是等计算量或无成本提升。

这个构造中能找到兼顾两类任务的权重，所以回放有明显帮助。真实任务是否改善、需要多少回放、是否存在目标冲突，需要另外实验。

In [12]:
for name, branch in forgetting_report["branches"].items():
    print(name, "参数更新次数：", branch["updates"],
          "每步新任务样本：", branch["target_examples_per_step"],
          "额外旧任务样本：", branch["replay_examples_per_step"],
          "基座不变：", branch["base_unchanged"])
print("这些不是大模型能力分数，也不是推理速度测量。")

target_only 参数更新次数： 200 每步新任务样本： 128 额外旧任务样本： 0 基座不变： True
with_replay 参数更新次数： 200 每步新任务样本： 128 额外旧任务样本： 128 基座不变： True
这些不是大模型能力分数，也不是推理速度测量。


## 5. 用开发保留集约束模型选择

只按目标开发 loss 选轮，可能选择旧任务大幅退步的 adapter。可以事先规定一条保留约束，例如：

$$A_{retain,dev}(\theta) \ge A_{retain,dev}(base)-0.02$$

在满足约束的候选里，再最小化目标开发 loss。注意这里的 0.02 是**两个百分点**，不是相对下降 2%。

本课的两个百分点只是教学设置。安全性、业务影响和评测噪声不同，不能用同一个阈值处理所有部署。

我们保留 step 0 作为候选；没有合格候选时应返回 `None`，不能强行选一个有明显退步的模型。

In [13]:
history = forgetting_report["branches"]["target_only"]["history"]
step = choose_dev_checkpoint(history, baseline_retention=1.0, max_drop=0.02)
print("只按新任务loss选择：", forgetting_report["branches"]["target_only"]["target_selected_step"])
print("带保留约束选择：", step)
row = forgetting_report["test"]["target_only_guarded_selected"]
print("锁定后的三个测试准确率：", {k:row[k]["accuracy"] for k in ("target","retention","shift")})
print("旧任务loss：", row["retention"]["loss"])
print("基座旧任务loss：", forgetting_report["test"]["base"]["retention"]["loss"])

只按新任务loss选择： 200
带保留约束选择： 37
锁定后的三个测试准确率： {'target': 1.0, 'retention': 1.0, 'shift': 1.0}
旧任务loss： 0.6810941963064723
基座旧任务loss： 0.14047042529152937


### 准确率通过约束，也不等于分布没有变化

上例可能保留 100% 的旧任务分类准确率，但旧任务交叉熵明显变差，预测距离决策边界更近。**一个保留指标无法代表所有能力、置信度或鲁棒性。**

更完善的方案应预先定义多个关键切片与指标，并考虑小样本波动。不能看完 test 结果后才不断改变保留阈值，再宣称 test 独立。

## 6. 评分时分开：缺失、运行失败、格式错、值错、停止异常

本课沿用第 35、36 课的严格约定：回答是仅有 `quantity` 键的 JSON；值是 0 到 9999 的整数，拒绝布尔值、重复键和额外解释。

答案正确但用尽预算，计入答案正确，却不计入本课的“严格成功”。这是固定任务协议，不是所有公开评测的统一定义。

**缺失和失败题不能从计划分母中消失。** 参考答案本身格式错误则报错，不能静默算作模型答错。

In [14]:
def quantity_value(text: str) -> int:
    obj = strict_json(text)
    require(isinstance(obj, dict) and set(obj) == {"quantity"}, "必须恰好有quantity字段")
    require(type(obj["quantity"]) is int and 0 <= obj["quantity"] <= 9999,
            "quantity必须是0..9999整数，排除布尔值；保持第35/36课规则")
    return obj["quantity"]


def score_quantity(response: dict | None, expected: str) -> dict:
    truth = quantity_value(expected)  # 参考答案本身错误应中止，不把它计成模型错误。
    if response is None:
        return {"coverage": False, "format_ok": False, "answer_correct": False,
                "strict_success": False, "failure": "missing"}
    require(isinstance(response, dict), "响应不是字典")
    if response.get("error") is not None:
        return {"coverage": False, "format_ok": False, "answer_correct": False,
                "strict_success": False, "failure": "runtime"}
    text = response.get("text")
    require(isinstance(text, str), "成功响应必须有字符串text")
    stop = response.get("stop_reason")
    require(stop in {"eos", "budget", "context"}, "成功响应必须明确停止原因")
    try:
        value = quantity_value(text)
    except (ValueError, TypeError, json.JSONDecodeError):
        return {"coverage": True, "format_ok": False, "answer_correct": False,
                "strict_success": False, "failure": "format"}
    correct = value == truth
    strict = correct and stop == "eos"
    failure = "ok" if strict else ("value" if not correct else "stopping")
    return {"coverage": True, "format_ok": True, "answer_correct": correct,
            "strict_success": strict, "failure": failure}


def index_rows(rows: list[dict], allowed: set[str]) -> dict[str, dict]:
    require(isinstance(rows, list), "响应集合必须是列表")
    result = {}
    for row in rows:
        require(isinstance(row, dict), "响应必须是字典")
        rid = row.get("id")
        require(isinstance(rid, str) and rid in allowed, "响应包含额外/非法ID")
        require(rid not in result, f"响应ID重复：{rid}")
        result[rid] = row
    return result


def validate_plan(plan: list[dict]) -> set[str]:
    require(isinstance(plan, list) and bool(plan), "计划题目为空")
    ids = set()
    for row in plan:
        require(isinstance(row, dict), "计划行必须是字典")
        for key in ("id", "group_id"):
            require(isinstance(row.get(key), str) and bool(row[key]), f"计划{key}非法")
        require(row["id"] not in ids, "计划ID重复")
        ids.add(row["id"])
    return ids


def paired_outcomes(plan: list[dict], before: list[dict], after: list[dict]) -> dict:
    ids = validate_plan(plan)
    a, b = index_rows(before, ids), index_rows(after, ids)
    for row in list(a.values())+list(b.values()):
        require(type(row.get("success")) is bool, "success必须是布尔值")
    groups = {"improved": [], "regressed": [], "both_correct": [], "both_wrong": []}
    rows = []
    for case in plan:
        rid = case["id"]
        x = a.get(rid, {}).get("success", False)
        y = b.get(rid, {}).get("success", False)
        kind = "both_correct" if x and y else "both_wrong" if not x and not y else "improved" if y else "regressed"
        groups[kind].append(rid)
        rows.append({"id": rid, "group_id": case["group_id"], "before": x, "after": y,
                     "delta": int(y)-int(x), "before_missing": rid not in a, "after_missing": rid not in b})
    n = len(plan)
    return {"planned": n, "base_rate": sum(r['before'] for r in rows)/n,
            "adapter_rate": sum(r['after'] for r in rows)/n,
            "delta": sum(r['delta'] for r in rows)/n, **groups, "rows": rows}


def percentile(values: list[float], probability: float) -> float:
    require(bool(values) and 0 <= probability <= 1, "分位数输入非法")
    require(all(math.isfinite(x) for x in values), "分位数不能含NaN/inf")
    s = sorted(values)
    p = (len(s)-1)*probability
    lo, hi = math.floor(p), math.ceil(p)
    return s[lo] + (s[hi]-s[lo])*(p-lo)


def paired_group_bootstrap(comparison: dict, repetitions=2000, seed=37) -> dict:
    require(type(repetitions) is int and repetitions >= 100, "重采样次数至少100")
    rows = comparison.get("rows")
    require(isinstance(rows, list) and bool(rows), "缺少配对行")
    by_group = defaultdict(list)
    seen = set()
    for row in rows:
        require(row["id"] not in seen, "重采样记录ID重复")
        seen.add(row["id"])
        require(type(row["before"]) is bool and type(row["after"]) is bool, "配对结果类型错误")
        by_group[row["group_id"]].append(int(row["after"])-int(row["before"]))
    summaries = [(sum(v), len(v)) for _, v in sorted(by_group.items())]
    point = sum(s for s, _ in summaries)/sum(n for _, n in summaries)
    count = len(summaries)
    result = {"point_delta": point, "source_groups": count, "confidence_level": 0.95,
              "repetitions": repetitions, "unit": "whole_source_group_paired",
              "estimand": "题目等权成功率之差；每次抽来源后重算题目总分母",
              "interval": None, "degenerate_observed_deltas": len({d for v in by_group.values() for d in v}) == 1,
              "limits": "近似percentile bootstrap；假设来源可独立代表目标总体；不覆盖训练种子/模型随机生成/分布变化/多重调参"}
    if count < 2:
        result["warning"] = "只有一个来源，不给出置信区间"
        return result
    rng = random.Random(seed)
    samples = []
    for _ in range(repetitions):
        draw = [summaries[rng.randrange(count)] for _ in range(count)]
        samples.append(sum(s for s, _ in draw)/sum(n for _, n in draw))
    result["interval"] = [percentile(samples, .025), percentile(samples, .975)]
    result["warning"] = ("来源数少，区间可能很不稳定；" if count < 20 else "") + \
        "即便区间退化为[0,0]也不能证明泛化效果完全相同；这是经验重采样，不是通过/失败判据"
    return result


def manual_pairing_demo() -> dict:
    plan = [{"id": f"q{i}", "group_id": f"g{i//2}"} for i in range(8)]
    before = [{"id": f"q{i}", "success": i < 6} for i in range(8)]
    after = [{"id": f"q{i}", "success": i in {0,1,2,3,4,6}} for i in range(8)]
    result = paired_outcomes(plan, before, after)
    return {"kind": "handwritten_scorer_test_not_model_scores", "paired": result,
            "uncertainty": paired_group_bootstrap(result)}

In [15]:
responses = [
    None,
    {"error":"CUDA out of memory"},
    {"text":'答案是 {"quantity":2}',"error":None,"stop_reason":"eos"},
    {"text":'{"quantity":3}',"error":None,"stop_reason":"eos"},
    {"text":'{"quantity":2}',"error":None,"stop_reason":"budget"},
    {"text":'{"quantity":2}',"error":None,"stop_reason":"eos"},
]
for response in responses:
    print(score_quantity(response, '{"quantity":2}'))

{'coverage': False, 'format_ok': False, 'answer_correct': False, 'strict_success': False, 'failure': 'missing'}
{'coverage': False, 'format_ok': False, 'answer_correct': False, 'strict_success': False, 'failure': 'runtime'}
{'coverage': True, 'format_ok': False, 'answer_correct': False, 'strict_success': False, 'failure': 'format'}
{'coverage': True, 'format_ok': True, 'answer_correct': False, 'strict_success': False, 'failure': 'value'}
{'coverage': True, 'format_ok': True, 'answer_correct': True, 'strict_success': False, 'failure': 'stopping'}
{'coverage': True, 'format_ok': True, 'answer_correct': True, 'strict_success': True, 'failure': 'ok'}


## 7. 同分数也可能有回归：按 ID 对齐两次结果

下面是明确手写的评分测试数据，不是任何模型的成绩。两次都答对 6/8，但修好一道的同时退步一道。

同一题配对能够保留题目难度的共同因素；只比较两个孤立总分会丢掉这些信息。[模型评测统计研究](https://www.anthropic.com/research/statistical-approach-to-model-evals)

In [16]:
manual_demo = manual_pairing_demo()
pair = manual_demo["paired"]
print("手工记录，不是模型：", manual_demo["kind"])
print("分数：", pair["base_rate"], "→", pair["adapter_rate"])
print("修好的题：", pair["improved"])
print("退步的题：", pair["regressed"])
print("未收到响应的题仍被当作失败。")

手工记录，不是模型： handwritten_scorer_test_not_model_scores
分数： 0.75 → 0.75
修好的题： ['q6']
退步的题： ['q5']
未收到响应的题仍被当作失败。


## 8. 有四个来源、每个两个问法，不能当成八个独立来源

统计不确定性时，相关问题不应被简单当作独立样本。很多评测包含同一篇文章的多个问题，需要保留这种成组结构。[同一研究中的成组分析](https://www.anthropic.com/research/statistical-approach-to-model-evals)

本课用**配对、按来源的 percentile bootstrap**作为教学工具：

1. 先把两次结果按题目 ID 对齐，得到每题变化 $d_i\in\{-1,0,1\}$。
2. 每次有放回抽取来源，抽中的来源带上全部问法与两路结果。
3. 在该次抽样中重新计算“总变化 / 总题数”。
4. 重复 2000 次，取 2.5% 和 97.5% 分位数。

组大小不同的时候，这一步是比率，不是直接平均每组的平均值。这里估计的是题目等权的成功率变化。

In [17]:
interval = manual_demo["uncertainty"]
print(json.dumps(interval, ensure_ascii=False, indent=2))
print("百分数点范围：", [100*x for x in interval["interval"]])

{
  "point_delta": 0.0,
  "source_groups": 4,
  "confidence_level": 0.95,
  "repetitions": 2000,
  "unit": "whole_source_group_paired",
  "estimand": "题目等权成功率之差；每次抽来源后重算题目总分母",
  "interval": [
    -0.375,
    0.375
  ],
  "degenerate_observed_deltas": false,
  "limits": "近似percentile bootstrap；假设来源可独立代表目标总体；不覆盖训练种子/模型随机生成/分布变化/多重调参",
  "warning": "来源数少，区间可能很不稳定；即便区间退化为[0,0]也不能证明泛化效果完全相同；这是经验重采样，不是通过/失败判据"
}
百分数点范围： [-37.5, 37.5]


### 必须保留的不确定性边界

只有四个来源，区间很不稳定；这不是发表级显著性论证。来源也未必是从目标总体随机抽样的。

本课区间不覆盖：训练随机种子、随机解码重复运行、模板设计变化、未来分布变化和反复调参。对同一批固定题目，它是近似的抽样敏感性分析，不是“模型真实能力的保证范围”。

如果所有观测变化都等于零，经验 bootstrap 只能抽出零，可能给出 `[0,0]`。**这不证明两个模型在未来任务上完全相同。** 只有一个来源时，本课直接不给区间。

In [18]:
tiny_plan = [{"id":str(i),"group_id":str(i)} for i in range(4)]
all_fail = paired_outcomes(tiny_plan, [], [])
empty_change = paired_group_bootstrap(all_fail)
print("全相同观测的区间：", empty_change["interval"])
print("是否退化：", empty_change["degenerate_observed_deltas"])
print("警告：", empty_change["warning"])

全相同观测的区间： [0.0, 0.0]
是否退化： True
警告： 来源数少，区间可能很不稳定；即便区间退化为[0,0]也不能证明泛化效果完全相同；这是经验重采样，不是通过/失败判据


## 9. 直接分析第 35、36 课保存的结果

现在把诊断方法接回真实学习项目。这个入口只读取 JSON/JSONL，不加载权重，不调用模型，也不执行原目录中的 Python 代码。

它会检查数据契约与指纹、来源隔离、两路请求编号和生成预算，使用原来的严格评分规则，从 `base_raw.jsonl` 与 `adapter_raw.jsonl` 重新计算结果。

**这两个旧课程默认只有目标任务测试集。** 仅凭它们不能证明没有旧能力退步或分布偏移问题。报告用 `null` 表示缺少证据，不填一个虚构的 0% 退步率。

同理，若第 36 课记录来自关闭 adapter 的量化基座，不能把它当作原始 FP16 基座的成绩。

In [19]:
def aggregate_quantity(records: list[dict], responses: list[dict]) -> dict:
    allowed = validate_plan(records)
    indexed = index_rows(responses,allowed)
    rows=[]
    for record in records:
        record_prompt(record)
        rid=record['id'];score=score_quantity(indexed.get(rid),record['messages'][-1]['content'])
        rows.append({"id":rid,"group_id":record['group_id'],**score})
    n=len(records)
    fields=('coverage','format_ok','answer_correct','strict_success')
    return {"planned":n,"received":len(responses),
            **{key+"_rate":sum(r[key] for r in rows)/n for key in fields},
            "failures":{k:sum(r['failure']==k for r in rows) for k in ('missing','runtime','format','value','stopping','ok')},
            "rows":rows}


def analyze_saved_run(directory, output_root="lesson_37_outputs") -> tuple[Path,dict]:
    source=Path(directory).resolve()
    require(source.is_dir(), "输入运行目录不存在")
    parts={split:read_jsonl(source/f"{split}_messages.jsonl") for split in ('train','dev','test')}
    contract=read_json(source/'contract.json')
    require(contract.get('format') in {'lesson35-lora-sft-v1','lesson36-qlora-v1'}, "只支持第35/36课本版数据契约")
    expected_hash = (contract.get('protocol', {}).get('split_sha256')
                     if contract['format'] == 'lesson35-lora-sft-v1'
                     else contract.get('data_sha256'))
    require(expected_hash==fingerprint(parts), "数据与保存的分割指纹不一致")
    audit=audit_splits(parts)
    require(not audit['hard_errors'], "跨划分精确提示重叠；先处理数据问题")
    base_raw=read_jsonl(source/'base_raw.jsonl');after_raw=read_jsonl(source/'adapter_raw.jsonl')
    planned=validate_plan(parts['test']);a=index_rows(base_raw,planned);b=index_rows(after_raw,planned)
    compatible=0
    for rid in a.keys() & b.keys():
        x,y=a[rid],b[rid]
        # 缺失/失败仍参与分母；可比较的原始请求必须指向同一输入。
        require(isinstance(x.get('request_sha256'),str) and len(x['request_sha256'])==64,
                "基座响应缺少有效请求指纹")
        require(x.get('request_sha256')==y.get('request_sha256'), "两路提示/模板token不一致")
        for response in (x,y):
            if 'input_ids' in response:
                require(fingerprint(response['input_ids'])==response['request_sha256'], "输入编号与请求指纹不一致")
        if x.get('error') is None and y.get('error') is None:
            require(type(x.get('max_new_tokens')) is int and x['max_new_tokens']>0 and
                    x['max_new_tokens']==y.get('max_new_tokens'), "两路生成预算不一致")
        compatible+=1
    base=aggregate_quantity(parts['test'],base_raw);after=aggregate_quantity(parts['test'],after_raw)
    pairs=paired_outcomes(parts['test'],
        [{"id":r['id'],"success":r['strict_success']} for r in base['rows'] if r['id'] in a],
        [{"id":r['id'],"success":r['strict_success']} for r in after['rows'] if r['id'] in b])
    history=read_json(source/'history.json') if (source/'history.json').exists() else None
    selection=read_json(source/'selection.json') if (source/'selection.json').exists() else None
    manifest=read_json(source/'manifest.json') if (source/'manifest.json').exists() else {}
    old_report=read_json(source/'report.json') if (source/'report.json').exists() else {}
    if old_report:
        for side, calculated in (("base",base),("adapter",after)):
            previous = old_report.get(side,{}).get("generation")
            if previous is not None:
                require(previous.get("planned")==calculated["planned"], "原报告计划分母不一致")
                for key in ("format_ok_rate","answer_correct_rate","strict_success_rate"):
                    require(previous.get(key)==calculated[key], f"重评分与保存报告不一致：{side}/{key}")
    result={"format":FORMAT,"kind":"reanalyzed_saved_responses_no_model_execution",
            "source_kind":manifest.get('kind',contract.get('kind')),
            "source_status":manifest.get('status','unknown'),"data_sha256":fingerprint(parts),
            "baseline":old_report.get('baseline','同一运行保存的关闭adapter基座；不要与其他运行的不同精度混比'),
            "compatible_request_pairs":compatible,"audit":audit,"base":base,"adapter":after,
            "paired":pairs,"paired_group_interval":paired_group_bootstrap(pairs),
            "selection":selection,"history":history,"retention_evidence":None,"shift_evidence":None,
            "conclusion":"只有目标测试集记录；不能据此断言无旧能力退化或无分布偏移问题",
            "limits":"请求指纹/数据签名是可复查线索，不是防恶意篡改证明；不加载或执行输入目录中的代码"}
    out=new_run(output_root,'analysis')
    write_json(out/'analysis.json',result)
    write_json(out/'manifest.json',{"format":FORMAT,"source_directory":str(source),
        "source_hashes":{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in source.glob('*') if p.is_file() and p.suffix in {'.json','.jsonl'}},
        "model_loaded":False,"source_modified":False})
    print("已重新评分：",out)
    print("严格成功率：",base['strict_success_rate'],"→",after['strict_success_rate'])
    print("旧能力证据：未提供；不自动填成零退化。")
    return out,result

In [20]:
ANALYZE_PREVIOUS_RUN = False
PREVIOUS_RUN = Path("lesson_36_outputs/请替换为实际运行目录")
if ANALYZE_PREVIOUS_RUN:
    analyzed_dir, previous_report = analyze_saved_run(PREVIOUS_RUN)
    print("报告：", analyzed_dir / "analysis.json")
else:
    print("未读取个人训练结果；把路径改为实际目录后再打开开关。")

未读取个人训练结果；把路径改为实际目录后再打开开关。


脚本方式：

```bash
python lesson_37_finetuning_diagnostics_and_generalization.py --analyze-run lesson_36_outputs/你的实际运行目录
```

第 35 课目录也支持。原目录不被修改，分析结果写到 `lesson_37_outputs/analysis_时间戳/`。如果只有一侧回答或文件损坏，需要先检查运行是否完成；程序不伪造缺少的输出。

实际制作时还用第 36 课代码执行了一轮离线训练，再读取真正保存的文件，核对了导入和重评分。这验证文件接口，不代表真实 Qwen 路径被执行。

## 10. 诊断后，怎样安排下一次实验？

不要把“加大学习率/多训几轮/扩大rank”当作所有问题的统一答案。先定位一类最可能的问题，控制其他变量，再测试一个改动。

| 观察 | 下一次可验证的对照 |
|---|---|
| 微小训练样本也过拟合不了 | 检查标签、移位、梯度、优化器参数对象；做少量样本记忆测试 |
| 训练很好、开发变差 | 检查噪声、来源划分；用开发集比较较早轮次和更少更新 |
| 格式错但值可能对 | 预先决定输出协议；开发侧补格式数据，不偷改测试评分器 |
| 新任务好、保留任务差 | 在训练侧比较回放、目标层/rank/学习率；开发保留集设置约束 |
| ID好、shift差 | 把分布差异写清；补有依据的训练变化，不声称单一OOD覆盖所有变化 |
| 基座与adapter全零分 | 先确认任务、模板、解码和评分器；零分差不能证明两者等效 |

回放会增加训练数据和计算；改变多个因素要做消融。使用过的最终测试材料以后可以保留作回归集，但新的结论需要新证据。

## 11. 独立练习

先不要看最后的参考答案。三个函数都应拒绝非法输入，不能修改传入对象。检查开关默认关闭，**运行全部单元不等于完成练习**。

A：同一 `group_id` 不能跨划分，同侧的多个问法可以存在；所有 ID 唯一。

B：按计划 ID 配对，把缺失回答视为失败；返回计划分母、两侧分数、改善/退步 ID。

C：只从开发历史里选择，先满足保留准确率约束，再按目标 loss 选最小，并列选最早；没有候选返回 `None`。

In [21]:
def check_exercise_a(fn):
    good={'train':[{'id':'a','group_id':'g1'},{'id':'b','group_id':'g1'}],
          'test':[{'id':'c','group_id':'g2'}]}
    before=copy.deepcopy(good)
    require(fn(good) is True,"合法按来源划分应返回True")
    require(good==before,"不能修改原数据")
    bad=copy.deepcopy(good);bad['test'][0]['group_id']='g1'
    expect_error(lambda:fn(bad))
    bad=copy.deepcopy(good);bad['test'][0]['id']='a'
    expect_error(lambda:fn(bad))
    print('练习A通过：同侧变体允许，跨侧同来源/重复ID拒绝，不改输入。')

def check_exercise_b(fn):
    plan=[{'id':str(i),'group_id':str(i//2)} for i in range(4)]
    a=[{'id':'0','success':True},{'id':'1','success':True},{'id':'2','success':False}]
    b=[{'id':'3','success':True},{'id':'1','success':True},{'id':'2','success':False}]
    saved=copy.deepcopy((plan,a,b));r=fn(plan,a,b)
    require(r['planned']==4 and r['base_rate']==.5 and r['adapter_rate']==.5,"分母必须包括缺失题")
    require(r['delta']==0 and r['improved']==['3'] and r['regressed']==['0'],"不能只报告总分或只报告改善")
    require((plan,a,b)==saved,"不得修改原数据")
    expect_error(lambda:fn(plan,a+a[:1],b))
    expect_error(lambda:fn(plan,a,b+[{'id':'extra','success':True}]))
    print('练习B通过：ID对齐、固定分母、修好与退步列表、重复/额外ID。')

def check_exercise_c(fn):
    history=[{'split':'dev','step':0,'new_loss':1.2,'retention_accuracy':1.},
             {'split':'dev','step':1,'new_loss':.8,'retention_accuracy':.99},
             {'split':'dev','step':2,'new_loss':.2,'retention_accuracy':.6},
             {'split':'dev','step':3,'new_loss':.8,'retention_accuracy':1.}]
    original=copy.deepcopy(history)
    require(fn(history,1.,.02)==1,"先满足保留约束，再最小loss；并列选早轮")
    require(history==original,"不得改写历史")
    bad=copy.deepcopy(history);bad[-1]['split']='test'
    expect_error(lambda:fn(bad,1.,.02))
    bad=copy.deepcopy(history);bad[-1]['new_loss']=float('nan')
    expect_error(lambda:fn(bad,1.,.02))
    require(fn([history[2]],1.,.02) is None,"没有合格候选应返回None，不能选不合格模型")
    print('练习C通过：dev限定、保留约束、并列规则、无候选与NaN处理。')

def expect_error(fn: Callable):
    try:
        fn()
    except (ValueError,TypeError,json.JSONDecodeError):
        return
    raise AssertionError('预期拒绝非法输入，但调用没有报错')

class Checks:
    def __init__(self): self.count=0
    def true(self,condition,name):
        if not bool(condition):raise AssertionError(name)
        self.count+=1
    def eq(self,a,b,name):self.true(a==b,name)
    def bad(self,fn,name):
        try: expect_error(fn)
        except AssertionError as exc:raise AssertionError(name) from exc
        self.count+=1

def demo_record(rid,gid,prompt,answer='{"quantity":1}'):
    return {'id':rid,'group_id':gid,'messages':[{'role':'user','content':prompt},{'role':'assistant','content':answer}]}

def legacy_fixture(root: Path, format_name='lesson36-qlora-v1') -> Path:
    """只用于接口测试：人为写出的记录，不是任何模型成绩。"""
    root.mkdir(exist_ok=True)
    parts={'train':[demo_record('tr','a','Training order A?')],
           'dev':[demo_record('dv','b','Development order B?')],
           'test':[demo_record('te0','c','Test order C?'),demo_record('te1','c','Please read order C.') ]}
    for split,rows in parts.items():
        (root/f'{split}_messages.jsonl').write_text(''.join(json.dumps(r)+'\n' for r in rows),encoding='utf-8')
    fixture_contract = {'format':format_name,'kind':'handwritten_interface_fixture'}
    if format_name == 'lesson35-lora-sft-v1':
        fixture_contract['protocol'] = {'split_sha256':fingerprint(parts)}
    else:
        fixture_contract['data_sha256'] = fingerprint(parts)
    write_json(root/'contract.json',fixture_contract)
    responses=[]
    for i,r in enumerate(parts['test']):
        ids=[10,i]
        responses.append({'id':r['id'],'input_ids':ids,'request_sha256':fingerprint(ids),'max_new_tokens':32,
            'text':'{"quantity":1}','stop_reason':'eos','error':None})
    for name in ('base','adapter'):
        (root/f'{name}_raw.jsonl').write_text(''.join(json.dumps(r)+'\n' for r in responses),encoding='utf-8')
    return root

def run_checks(verbose=True) -> int:
    c=Checks()
    c.eq(fingerprint({'a':1,'b':2}),fingerprint({'b':2,'a':1}),'规范JSON指纹')
    c.bad(lambda:strict_json('{"a":1,"a":2}'),'重复键')
    c.bad(lambda:strict_json('{"a":NaN}'),'非有限数')
    c.eq(quantity_value(' {"quantity":2} '),2,'严格数量')
    for text in ('{"quantity":true}','{"quantity":"2"}','{"quantity":2,"x":0}','{"quantity":1,"quantity":2}','text {"quantity":2}','{"quantity":-1}','{"quantity":10000}'):
        c.bad(lambda text=text:quantity_value(text),'错误JSON数量')
    rec=demo_record('x','a','Question','{"quantity":2}')
    c.eq(len(record_prompt(rec)),1,'去掉最后答案')
    changed=copy.deepcopy(rec);changed['messages'][-1]['content']='{"quantity":3}'
    c.eq(record_prompt(rec),record_prompt(changed),'改变答案不改变请求')
    c.bad(lambda:record_prompt({'id':'x','group_id':'a','messages':[]}), '空对话')
    parts={'train':[rec], 'test':[demo_record('y','b','Another question')]}
    c.true(group_disjoint(parts),'来源不重叠')
    c.bad(lambda:group_disjoint({'train':[rec],'test':[demo_record('y','a','Other')]}),'来源泄漏')
    c.bad(lambda:group_disjoint({'train':[rec],'test':[demo_record('x','b','Other')]}),'跨侧重复ID')
    c.true(not audit_splits(parts)['hard_errors'],'合法切分')
    overlap={'train':[rec],'test':[demo_record('z','b','Question','{"quantity":3}')]}
    c.true(audit_splits(overlap)['hard_errors'][0]['conflicting_answer'],'不同来源但相同提示/冲突答案')
    pair={'train':[demo_record('a','a','Order  12')],'test':[demo_record('b','b','Order 12')]}
    saved=copy.deepcopy(pair);ar=audit_splits(pair)
    c.true(ar['candidate_count']==1 and not ar['hard_errors'],'归一化仅警告')
    c.eq(pair,saved,'审计不改源文本')
    c.eq(normalized_prompt_for_audit([{'role':'user','content':'e\u0301'}]),normalized_prompt_for_audit([{'role':'user','content':'é'}]),'NFC审计')
    leak=leakage_demo()
    c.true(leak['bad_split_detected'],'泄漏示例可被发现')
    c.eq(leak['overlapping_sources']['accuracy'],1.,'重叠来源记忆')
    c.eq(leak['unseen_sources']['accuracy'],.5,'未见来源常数基线')
    correct={'text':'{"quantity":1}','stop_reason':'eos','error':None}
    c.true(score_quantity(correct,'{"quantity":1}')['strict_success'],'正确且EOS')
    c.eq(score_quantity(None,'{"quantity":1}')['failure'],'missing','缺失分类')
    c.eq(score_quantity({'error':'OOM'},'{"quantity":1}')['failure'],'runtime','运行失败分类')
    c.eq(score_quantity(dict(correct,text='1'),'{"quantity":1}')['failure'],'format','格式错误分类')
    c.eq(score_quantity(dict(correct,text='{"quantity":2}'),'{"quantity":1}')['failure'],'value','值错误分类')
    budget=score_quantity(dict(correct,stop_reason='budget'),'{"quantity":1}')
    c.true(budget['answer_correct'] and not budget['strict_success'],'答案正确但未正常结束')
    c.bad(lambda:score_quantity(correct,'{"quantity":true}'),'非法参考答案必须报错')
    c.bad(lambda:score_quantity(dict(correct,stop_reason='unknown'),'{"quantity":1}'),'未知停止标志')
    check_exercise_a(answer_a);c.count+=1
    check_exercise_b(answer_b);c.count+=1
    check_exercise_c(answer_c);c.count+=1
    demo=manual_pairing_demo();pa=demo['paired']
    c.eq(pa['base_rate'],.75,'手工评分基准')
    c.eq(pa['adapter_rate'],.75,'同分')
    c.eq((len(pa['improved']),len(pa['regressed'])),(1,1),'同分掩盖回归')
    c.eq(demo['uncertainty']['source_groups'],4,'以来源计数')
    c.eq(paired_group_bootstrap(pa),paired_group_bootstrap(pa),'固定bootstrap随机种子')
    c.eq(percentile([0.,10.],.25),2.5,'线性分位数')
    c.bad(lambda:percentile([], .5),'空分位数')
    c.bad(lambda:paired_group_bootstrap(pa,repetitions=1),'错误重采样次数')
    single=paired_outcomes([{'id':'x','group_id':'g'}],[{'id':'x','success':False}],[{'id':'x','success':True}])
    c.true(paired_group_bootstrap(single)['interval'] is None,'单来源不报区间')
    constant=paired_outcomes([{'id':str(i),'group_id':str(i)} for i in range(3)],[],[])
    ci=paired_group_bootstrap(constant)
    c.eq(ci['interval'],[0.,0.],'全相同观测退化区间')
    c.true(ci['degenerate_observed_deltas'],'标记退化而非宣布等价')
    c.bad(lambda:paired_outcomes([{'id':'x','group_id':'g'}],[{'id':'x','success':1}],[]),'success拒绝整数冒充bool')
    # 组大小不同的bootstrap采用ratio，不把平均组差当作整体题目差。
    uneven_plan=[{'id':str(i),'group_id':'big' if i<9 else 'small'} for i in range(10)]
    uneven=paired_outcomes(uneven_plan,[],[{'id':str(i),'success':i<9} for i in range(10)])
    c.eq(paired_group_bootstrap(uneven)['point_delta'],.9,'不等组大小的题目加权')
    rows=[{'split':'dev','step':0,'new_loss':2.,'retention_accuracy':1.},
          {'split':'dev','step':1,'new_loss':1.,'retention_accuracy':.5}]
    c.eq(choose_dev_checkpoint(rows),0,'基座可以保留')
    c.bad(lambda:choose_dev_checkpoint([dict(rows[0],split='test')]),'拒绝测试集选轮')
    c.bad(lambda:choose_dev_checkpoint([rows[0],rows[0]]),'重复步数')
    c.bad(lambda:choose_dev_checkpoint([dict(rows[0],new_loss=float('inf'))]),'非有限loss')
    c.bad(lambda:choose_dev_checkpoint(rows,max_drop=-1),'负退步阈值')
    x,y=noisy_xor(1,12);x2,y2=noisy_xor(1,12)
    c.true(torch.equal(x,x2) and torch.equal(y,y2),'数据复现')
    c.eq(list(x.shape),[12,2],'数据形状')
    m=SharedLoRA();data=domain_data(1,12,'target')
    c.true(torch.equal(m(data[0]),m(data[0],True)),'B=0初始恒等')
    c.eq(sum(p.numel() for p in m.parameters()),3,'仅A/B三参数，无节省比例声称')
    c.true('base_weight' in dict(m.named_buffers()),'基座为冻结buffer')
    frozen=tensor_hash(m.base_weight);s=state_copy(m)
    opt=torch.optim.Adam(m.parameters(),lr=.03)
    loss=F.binary_cross_entropy_with_logits(m(data[0]),data[1]);loss.backward()
    c.eq(float(m.a.grad.abs().max()),0.,'A首步梯度零')
    c.true(float(m.b.grad.abs().max())>0,'B首步梯度非零')
    opt.step()
    c.eq(frozen,tensor_hash(m.base_weight),'更新不改基座')
    c.true(torch.equal(s['b'],torch.zeros_like(s['b'])),'快照必须复制')
    c.true(torch.equal(m(data[0],True),F.linear(data[0],m.base_weight)),'关闭adapter还原')
    clone=SharedLoRA();clone.load_state_dict(state_copy(m))
    c.true(torch.equal(m(data[0]),clone(data[0])),'重载输出')
    c.bad(lambda:domain_data(1,3,'unknown'),'未知域')
    c.bad(lambda:fit_adapter_branch({'test':data},False,1),'训练拒绝test')
    check_model=make_classifier();check_model.train()
    classification_metrics(check_model,noisy_xor(1,4))
    c.true(check_model.training,'验证恢复模式')
    c.true(all(p.grad is None for p in check_model.parameters()),'验证不产生梯度')
    # 真正读写第35/36课格式的接口测试；明确只有手工数据。
    with tempfile.TemporaryDirectory() as tmp:
        root=Path(tmp);src=legacy_fixture(root/'source')
        hashes={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in src.iterdir()}
        _,report=analyze_saved_run(src,root/'out')
        c.eq(report['base']['strict_success_rate'],1.,'原始记录重新评分')
        c.true(report['retention_evidence'] is None,'缺失旧能力证据不能编造')
        c.eq(report['paired_group_interval']['interval'],None,'单来源导入区间为空')
        c.eq({p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in src.iterdir()},hashes,'导入不改原目录')
        contract=read_json(src/'contract.json');contract['data_sha256']='bad';write_json(src/'contract.json',contract)
        c.bad(lambda:analyze_saved_run(src,root/'out'),'数据指纹被篡改')
        src2=legacy_fixture(root/'other');rs=read_jsonl(src2/'adapter_raw.jsonl');rs[0]['max_new_tokens']=64
        (src2/'adapter_raw.jsonl').write_text(''.join(json.dumps(r)+'\n' for r in rs),encoding='utf-8')
        c.bad(lambda:analyze_saved_run(src2,root/'out'),'两路生成预算不一致')
        src35 = legacy_fixture(root/'source35', 'lesson35-lora-sft-v1')
        _, report35 = analyze_saved_run(src35, root/'out')
        c.eq(report35['base']['strict_success_rate'], 1., '第35课嵌套数据契约兼容')
    if verbose:print(f'固定检查通过：{c.count} 项；不是LLM/GPU实测。')
    return c.count

In [22]:
def group_disjoint_ex(parts):
    # 检查完合法数据后返回True；非法数据抛出ValueError。
    raise NotImplementedError("请完成练习A")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    check_exercise_a(group_disjoint_ex)

In [23]:
def paired_outcomes_ex(plan, before, after):
    # before/after各行为 {"id":..., "success":True/False}。
    # 返回字段至少有 planned, base_rate, adapter_rate, delta, improved, regressed。
    raise NotImplementedError("请完成练习B")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    check_exercise_b(paired_outcomes_ex)

In [24]:
def choose_checkpoint_ex(history, baseline_retention=1.0, max_drop=0.02):
    # 每行含 split, step, new_loss, retention_accuracy。
    raise NotImplementedError("请完成练习C")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    check_exercise_c(choose_checkpoint_ex)

## 12. 参考实现与验收

先保存自己的答案，再看这一节。参考函数使用独立名称，不覆盖练习函数。

为了避免只背包装函数，请展开前面对应实现，自己解释每条校验以及分母、并列选择的规则。

In [25]:
def answer_a(parts: dict[str, list[dict]]) -> bool:
    require(isinstance(parts, dict) and len(parts) >= 2, "至少两个划分")
    ownership, ids = {}, set()
    for split, records in parts.items():
        require(isinstance(records, list) and bool(records), "划分不能为空")
        for row in records:
            require(isinstance(row, dict), "记录不是字典")
            rid, gid = row.get("id"), row.get("group_id")
            require(isinstance(rid, str) and bool(rid) and isinstance(gid, str) and bool(gid), "ID/来源非法")
            require(rid not in ids, f"重复记录ID：{rid}")
            ids.add(rid)
            require(gid not in ownership or ownership[gid] == split,
                    f"同一来源跨划分：{gid}")
            ownership[gid] = split
    return True

check_exercise_a(answer_a)

练习A通过：同侧变体允许，跨侧同来源/重复ID拒绝，不改输入。


In [26]:
def answer_b(plan: list[dict], before: list[dict], after: list[dict]) -> dict:
    ids = validate_plan(plan)
    a, b = index_rows(before, ids), index_rows(after, ids)
    for row in list(a.values())+list(b.values()):
        require(type(row.get("success")) is bool, "success必须是布尔值")
    groups = {"improved": [], "regressed": [], "both_correct": [], "both_wrong": []}
    rows = []
    for case in plan:
        rid = case["id"]
        x = a.get(rid, {}).get("success", False)
        y = b.get(rid, {}).get("success", False)
        kind = "both_correct" if x and y else "both_wrong" if not x and not y else "improved" if y else "regressed"
        groups[kind].append(rid)
        rows.append({"id": rid, "group_id": case["group_id"], "before": x, "after": y,
                     "delta": int(y)-int(x), "before_missing": rid not in a, "after_missing": rid not in b})
    n = len(plan)
    return {"planned": n, "base_rate": sum(r['before'] for r in rows)/n,
            "adapter_rate": sum(r['after'] for r in rows)/n,
            "delta": sum(r['delta'] for r in rows)/n, **groups, "rows": rows}

check_exercise_b(answer_b)

练习B通过：ID对齐、固定分母、修好与退步列表、重复/额外ID。


In [27]:
def answer_c(history: list[dict], baseline_retention=1.0, max_drop=0.02) -> int | None:
    require(isinstance(history, list) and bool(history), "历史为空")
    require(math.isfinite(baseline_retention) and 0 <= baseline_retention <= 1, "基准保留准确率无效")
    require(math.isfinite(max_drop) and 0 <= max_drop <= 1, "退步阈值无效")
    steps, eligible = set(), []
    for row in history:
        require(row.get("split") == "dev", "只能用dev选择；不能把train/test/OOD作为选轮输入")
        step, loss, retain = row.get("step"), row.get("new_loss"), row.get("retention_accuracy")
        require(type(step) is int and step >= 0 and step not in steps, "step重复/非法")
        steps.add(step)
        require(type(loss) in (int,float) and math.isfinite(loss) and loss >= 0, "新任务dev loss无效")
        require(type(retain) in (int,float) and math.isfinite(retain) and 0 <= retain <= 1, "保留集准确率无效")
        if retain >= baseline_retention-max_drop:
            eligible.append((loss, step))
    return min(eligible)[1] if eligible else None

check_exercise_c(answer_c)

练习C通过：dev限定、保留约束、并列规则、无候选与NaN处理。


## 13. 自动检查与完整导出

下面会运行评分、来源审计、数据选择和文件接口检查。接口测试使用临时手工记录，并明确标为测试数据；不是模型成绩。

默认脚本和 Notebook 使用相同实现。完整导出再独立运行一遍两个小实验，用来检查前面观察到的结果可以被重新生成。

In [28]:
checks_count = run_checks()
print("本次通过检查数：", checks_count)

练习A通过：同侧变体允许，跨侧同来源/重复ID拒绝，不改输入。
练习B通过：ID对齐、固定分母、修好与退步列表、重复/额外ID。
练习C通过：dev限定、保留约束、并列规则、无候选与NaN处理。
已重新评分： /tmp/tmp6yzhmlzt/out/analysis_20261102T054554_675707Z
严格成功率： 1.0 → 1.0
旧能力证据：未提供；不自动填成零退化。
已重新评分： /tmp/tmp6yzhmlzt/out/analysis_20261102T054554_680415Z
严格成功率： 1.0 → 1.0
旧能力证据：未提供；不自动填成零退化。
固定检查通过：76 项；不是LLM/GPU实测。
本次通过检查数： 76


In [29]:
def run_all(output_root="lesson_37_outputs") -> tuple[Path,dict]:
    out=new_run(output_root)
    manifest={"format":FORMAT,"status":"running","environment":{"python":platform.python_version(),
        "torch":torch.__version__,"device":"cpu","threads":1,"cuda_executed":False},
        "sources":SOURCES,"limitations":"实际训练的小模型和明确标记的手工评分样例；不是Qwen/3060结果"}
    write_json(out/'manifest.json',manifest)
    previous_threads=torch.get_num_threads();torch.set_num_threads(1)
    try:
        leakage=leakage_demo();scoring=manual_pairing_demo()
        overfit,overfit_states=fit_overfit_demo()
        forgetting,adapter_states=forgetting_demo()
        torch.save({"format":FORMAT,"overfit":overfit_states,"adapters":adapter_states},out/'teaching_models.pt')
        loaded=torch.load(out/'teaching_models.pt',map_location='cpu',weights_only=True)
        for label,state in overfit_states.items():
            require(all(torch.equal(t,loaded['overfit'][label][k]) for k,t in state.items()),"重载状态不一致")
        for branch,selections in adapter_states.items():
            for label,state in selections.items():
                m=SharedLoRA();m.load_state_dict(loaded['adapters'][branch][label],strict=True)
                require(all(torch.equal(t,m.state_dict()[k]) for k,t in state.items()),"adapter重载状态不一致")
        result={"leakage":leakage,"scoring_demo":scoring,"overfit":overfit,"forgetting":forgetting,
                "roundtrip_state_exact":True}
        write_json(out/'leakage.json',leakage);write_json(out/'scoring_demo.json',scoring)
        write_json(out/'overfit.json',overfit);write_json(out/'forgetting.json',forgetting)
        write_json(out/'experiment.json',result)
        manifest['status']='completed';write_json(out/'manifest.json',manifest)
        print("完成：",out)
        print("过拟合实验：dev选择步数",overfit['selected_step'],"，最终训练loss",overfit['history'][-1]['train_loss'])
        for label in ('base','target_only_target_selected','with_replay_target_selected'):
            r=forgetting['test'][label]
            print(label,{s:r[s]['accuracy'] for s in ('target','retention','shift')})
        print("仅CPU教学结果；没有实际LLM或GPU能力/性能结论。")
        return out,result
    except Exception as exc:
        manifest.update(status='failed',error=f'{type(exc).__name__}: {exc}')
        write_json(out/'manifest.json',manifest)
        raise
    finally:
        torch.set_num_threads(previous_threads)

In [30]:
RUN_FULL_EXPORT = True
if RUN_FULL_EXPORT:
    run_dir, full_report = run_all()
    assert full_report["overfit"] == overfit_report
    assert full_report["forgetting"] == forgetting_report
    print("独立重新运行与前面两项实验报告完全一致。")
    print("总报告：", run_dir / "experiment.json")
else:
    print("本次未导出完整报告。")

完成： lesson_37_outputs/offline_20261102T054554_712326Z
过拟合实验：dev选择步数 10 ，最终训练loss 0.0009719692170619965
base {'target': 0.0, 'retention': 1.0, 'shift': 1.0}
target_only_target_selected {'target': 1.0, 'retention': 0.0, 'shift': 0.0}
with_replay_target_selected {'target': 1.0, 'retention': 1.0, 'shift': 1.0}
仅CPU教学结果；没有实际LLM或GPU能力/性能结论。
独立重新运行与前面两项实验报告完全一致。
总报告： lesson_37_outputs/offline_20261102T054554_712326Z/experiment.json


### 脚本命令与输出

```bash
# 只运行检查
python lesson_37_finetuning_diagnostics_and_generalization.py --checks-only

# 来源与评分机制，不训练神经网络
python lesson_37_finetuning_diagnostics_and_generalization.py --audit-only

# 完整离线训练、诊断与保存
python lesson_37_finetuning_diagnostics_and_generalization.py
```

每次创建新目录，不覆盖以前运行：

```text
lesson_37_outputs/offline_时间戳/
    manifest.json
    leakage.json
    scoring_demo.json
    overfit.json
    forgetting.json
    experiment.json
    teaching_models.pt
```

`teaching_models.pt` 是教学模型参数快照，不是包含优化器、随机状态和游标的精确断点续训文件。读取 `.pt` 只使用自己生成的可信文件。文件分析目录保留原始来源指纹，但哈希并不能防止有意伪造整份记录。

本课未训练或调用真实LLM，也未进行3060显存/速度测试。

## 14. 最终验收：用自己的话回答

1. 为什么训练 loss 接近零，也可能不应该保留最后一轮？
2. 为什么基座哈希不变，开启 LoRA 后的旧任务仍可能退步？
3. 为什么两个模型同分数，还需要逐题配对？
4. 两种问法来自同一文档，bootstrap 时为什么一起抽取？
5. 第 36 课目标任务只评了八条且两路全失败，为什么不能得出“量化没伤害能力”或“微调没有影响”？
6. 一条保留准确率约束通过后，旧任务 loss 变坏意味着还要查什么？

**下一课：第 38 课——RAG 基础：文本分块、词法检索与证据引用。** 从训练参数转向给模型提供外部资料，先建立可检查的检索基线，再逐步引入 embedding 检索与生成。

## 本次制作验证记录

- CPU 环境：Python 3.13.5、PyTorch 2.10.0+cpu。
- 30 个代码单元在全新内核中顺序执行，固定检查 76 项通过。
- 三个练习参考实现分别在独立进程中验证；六种故障注入被固定检查发现。
- Notebook 与独立脚本的实验报告、教学模型张量一致。
- 使用第 35、36 课真实代码各执行一轮离线训练，读取实际保存的文件并重评分；这不是预训练 Qwen 实测。
- 未执行真实 LLM 推理或 CUDA，不提供 3060 显存、速度或真实模型能力数值。

执行成功和完成独立练习是两回事；实际运行的具体日志保留在前面的代码输出中。

## 参考资料

- [scikit-learn：数据泄漏与常见陷阱](https://scikit-learn.org/stable/common_pitfalls.html)
- [scikit-learn：按来源切分](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html)
- [Scaling Laws for Forgetting When Fine-Tuning Large Language Models](https://arxiv.org/abs/2401.05605)
- [Anthropic：评测统计、来源相关性与配对差异](https://www.anthropic.com/research/statistical-approach-to-model-evals)
- [PyTorch：可复现性边界](https://docs.pytorch.org/docs/stable/notes/randomness.html)
- [Python JSON 文档](https://docs.python.org/3/library/json.html)

资料为方法来源；本讲义中的训练数值由随附代码实际产生，不是从论文抄来的模型性能。